In [3]:
import os, sys
import pyspark


#아래의 설정은 jupyter notebook의 임시적으로 설정하는 방법이고
#영구적으로 아래와 같은 설정을 하기 위해서는 환경변수를 설정해줘야 한다


os.environ["PYSPARK_PYTHON"]=sys.executable          # Worker가 사용할 Python 실행 파일 경로 설정 (리눅스 "/usr/bin/python3")
os.environ["PYSPARK_DRIVER_PYTHON"]=sys.executable   # Driver에서도 동일한 Python 경로 설정
#os.environ['HADOOP_HOME']=os.getcwd() # 현재 디렉터리를 HADOOP_HOME으로 설정
#os.environ["PATH"] += os.path.join(os.environ['HADOOP_HOME'], 'bin') # PATH에 Hadoop 바이너리 추가

#driver랑 worker랑 python버전을 일치시키는 방법
#참고로 driver가 worker에게 지시를 내리기 때문에 두개의 버전을 일치시키는 것이 필요하다

myConf=pyspark.SparkConf() # 기본 설정 객체 생성, 여기에 필요한 설정 정의
#myConf=pyspark.SparkConf().set("spark.driver.bindAddress", "127.0.0.1") 드라이버 바인딩 주소 설정
#myConf=pyspark.SparkConf().set("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.1.1") 



###############SparkSessin 매우 중요##################

#spark를 시작할 땐 미리 아래의 코드를 생성해두는 것이 좋다

spark = pyspark.sql.SparkSession\
    .builder\
    .master("local")\
    .appName("myApp")\
    .config(conf=myConf)\
    .getOrCreate()

#마지막의 getOrcreate는 메모리를 효율적으로 관리할 수 있게 한다

In [4]:
myList=[1,2,3,4,5,6,7]

In [5]:
#sparkContext 는 RDD를 사용할 때 이용하는 context 
#지금은 session으로 쓰는중

myRdd1 = spark.sparkContext.parallelize(myList)

##plus)) streamingcontext는 실시간

In [6]:
myRdd1.take(3)

[1, 2, 3]

In [7]:
spark.sparkContext.parallelize([0, 2, 3, 4, 6], 2).glom().collect()

[[0, 2], [3, 4, 6]]

In [8]:
%%writefile data/ds_spark_wiki.txt
Wikipedia
Apache Spark is an open source cluster computing framework.
아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.
Apache Spark Apache Spark Apache Spark Apache Spark
아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크
Originally developed at the University of California, Berkeley's AMPLab,
the Spark codebase was later donated to the Apache Software Foundation,
which has maintained it since.
Spark provides an interface for programming entire clusters with
implicit data parallelism and fault-tolerance.

Overwriting data/ds_spark_wiki.txt


In [13]:
from pyspark.sql import SparkSession

myRdd2=spark.sparkContext\
    .textFile(os.path.join("data","ds_spark_wiki.txt"))

In [14]:
myRdd2.first()

'Wikipedia'

In [15]:
myRdd2.take(3)

['Wikipedia',
 'Apache Spark is an open source cluster computing framework.',
 '아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.']

In [28]:
#dataframe 자료형

In [29]:
import os
myDf=spark.read.text(os.path.join("data", "ds_spark_wiki.txt"))

In [30]:
print (myDf.first())

Row(value='Wikipedia')


In [31]:
print (type(myDf))

<class 'pyspark.sql.dataframe.DataFrame'>


In [32]:
#RDD 자료형

In [33]:
%%writefile ./data/ds_spark_2cols.csv
35, 2
40, 27
12, 38
15, 31
21, 1
14, 19
46, 1
10, 34
28, 3
48, 1
16, 2
30, 3
32, 2
48, 1
31, 2
22, 1
12, 3
39, 29
19, 37
25, 2

Writing ./data/ds_spark_2cols.csv


In [34]:
myRdd4 = spark.sparkContext\
    .textFile(os.path.join("data","ds_spark_2cols.csv"))

In [35]:
myList=myRdd4.take(5)

print(myList)

['35, 2', '40, 27', '12, 38', '15, 31', '21, 1']


In [36]:
myRdd4.map(lambda x: x.split(","))\
    .map(lambda x: int(x[0])+int(x[1]))\
    .collect()

[37,
 67,
 50,
 46,
 22,
 33,
 47,
 44,
 31,
 49,
 18,
 33,
 34,
 49,
 33,
 23,
 15,
 68,
 56,
 27]

In [52]:
#스파크는 HDFS를 사용한다 HD :: 하둡, FS :: file system

In [54]:
#ex ) mapreduce가 체이닝으로 연결이 되는 것

In [60]:
#RDD는 mutable => data의 원본은 수정하지 못하는 성격을 가지고 있다.
#수정한 것 처럼 보이지만 사실 복사본을 수정하는 것

#이는 추후 통합되거나 병합될 수 있어서 죽으면 안 될때, 안정성을 생각할 때 주로 사용한다
#back up 할 때 사용하는 것 그래서 다시 불러올 때 빨리 불러올수록 성능이 좋은 것으로 본다

In [103]:
# transformation => 직접 계산하지 않고 -> action으로 넘김 (RDD를 값으로 변환시키는 것)          =>lazy연산   =>메모리의 효율성때문에 ~

In [66]:
#2. 파일을 읽고 RDD 생성하기

In [16]:
popRdd = spark.sparkContext\
    .textFile(os.path.join("data","경기도 의정부시_인구현황_20240930.csv"), use_unicode=True)

In [17]:
popRdd.take(5)

['�������,�α���(��),�α���(��),�α���(��),������(��),������(��),������(��),����,�����,������α�,���������,�����μ���,�μ���ȭ��ȣ,�����ͱ�������',
 '������1��,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,��\u2d75 �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30',
 '������2��,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,��\u2d75 �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30',
 'ȣ��1��,34059,16442,17617,7.40,3.57,3.83,93.33,15178,2.24,��\u2d75 �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30',
 'ȣ��2��,32529,15643,16886,7.07,3.40,3.67,92.64,13272,2.45,��\u2d75 �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30']

In [18]:
for i in popRdd.take(5):
    print(i)

�������,�α���(��),�α���(��),�α���(��),������(��),������(��),������(��),����,�����,������α�,���������,�����μ���,�μ���ȭ��ȣ,�����ͱ�������
������1��,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
������2��,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
ȣ��1��,34059,16442,17617,7.40,3.57,3.83,93.33,15178,2.24,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
ȣ��2��,32529,15643,16886,7.07,3.40,3.67,92.64,13272,2.45,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30


In [19]:
for i in popRdd.take(10):
    print(i)

�������,�α���(��),�α���(��),�α���(��),������(��),������(��),������(��),����,�����,������α�,���������,�����μ���,�μ���ȭ��ȣ,�����ͱ�������
������1��,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
������2��,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
ȣ��1��,34059,16442,17617,7.40,3.57,3.83,93.33,15178,2.24,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
ȣ��2��,32529,15643,16886,7.07,3.40,3.67,92.64,13272,2.45,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
��ϵ�,18741,8923,9818,4.07,1.94,2.13,90.88,8235,2.28,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
�Ű�1��,39747,19449,20298,8.64,4.23,4.41,95.82,17136,2.32,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
�Ű�2��,44657,21531,23126,9.70,4.68,5.02,93.10,18826,2.37,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
�ۻ�1��,30633,14946,15687,6.66,3.25,3.41,95.28,13469,2.27,��⵵ �����ν�û,�ο����ǰ�,031-828-2466,2024-09-30
�ۻ�2��,31499,15582,15917,6.84,3.39,3.46

In [21]:
agedRdd = spark.sparkContext\
    .textFile(os.path.join("data","제주특별자치도 서귀포시_고령화비율및노령화지수현황_20240419.csv"), use_unicode=True)

In [22]:
for i in agedRdd.take(5):
    print(i)

�⵵,��,�������� �α���,65���̻� �α��� ,14������ �α���,���ȭ����,���ȭ����,�����ͱ�������
2008,12,153120,22241,26792,14.53,83.01,2024-04-19
2009,12,152285,23031,25504,15.12,90.30,2024-04-19
2010,12,153716,23990,24633,15.61,97.39,2024-04-19
2011,12,153366,24839,23686,16.20,104.87,2024-04-19


In [23]:
for i in agedRdd.take(5)[1:]:
    print(i)

2008,12,153120,22241,26792,14.53,83.01,2024-04-19
2009,12,152285,23031,25504,15.12,90.30,2024-04-19
2010,12,153716,23990,24633,15.61,97.39,2024-04-19
2011,12,153366,24839,23686,16.20,104.87,2024-04-19


In [24]:
for i in agedRdd.take(10)[1:]:
    print(i)

2008,12,153120,22241,26792,14.53,83.01,2024-04-19
2009,12,152285,23031,25504,15.12,90.30,2024-04-19
2010,12,153716,23990,24633,15.61,97.39,2024-04-19
2011,12,153366,24839,23686,16.20,104.87,2024-04-19
2012,12,154057,25826,22861,16.76,112.97,2024-04-19
2013,12,155641,26936,22393,17.31,120.29,2024-04-19
2014,12,158512,27877,22058,17.59,126.38,2024-04-19
2015,12,164519,28979,22362,17.61,129.59,2024-04-19
2016,12,170932,30030,23044,17.57,130.32,2024-04-19


In [25]:
for i in agedRdd.take(10)[6:]:
    print(i)

2013,12,155641,26936,22393,17.31,120.29,2024-04-19
2014,12,158512,27877,22058,17.59,126.38,2024-04-19
2015,12,164519,28979,22362,17.61,129.59,2024-04-19
2016,12,170932,30030,23044,17.57,130.32,2024-04-19


In [78]:
for i in agedRdd.take(5)[0:]:
    print(i)

�⵵,��,�������� �α���,65���̻� �α��� ,14������ �α���,���ȭ����,���ȭ����,�����ͱ�������
2008,12,153120,22241,26792,14.53,83.01,2024-04-19
2009,12,152285,23031,25504,15.12,90.30,2024-04-19
2010,12,153716,23990,24633,15.61,97.39,2024-04-19
2011,12,153366,24839,23686,16.20,104.87,2024-04-19


In [79]:
#위와같이 시작하는 line을 지정할 수 있다

In [80]:
#RDD는 mapreduce 함수에 최적화되어있다!

In [81]:
#인코딩 추가! ==> 바이너리 형식 (binaryFiles) 이렇게 읽을 땐 decode를 곡 해줘야 한다 (코드 -> 문자 일 때 decode라고 한다)

In [26]:
popRddBin = spark.sparkContext.binaryFiles(os.path.join("data","경기도 의정부시_인구현황_20240930.csv"))

In [27]:
_my = popRddBin.map(lambda x :x[1].decode('euc-kr')) # "utf-8"

In [29]:
_my.take(10)

['행정기관,인구수(계),인구수(남),인구수(여),구성비(계),구성비(남),구성비(여),성비,세대수,세대당인구,관리기관명,관리부서명,부서전화번호,데이터기준일자\r\n의정부1동,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n의정부2동,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n호원1동,34059,16442,17617,7.40,3.57,3.83,93.33,15178,2.24,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n호원2동,32529,15643,16886,7.07,3.40,3.67,92.64,13272,2.45,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n장암동,18741,8923,9818,4.07,1.94,2.13,90.88,8235,2.28,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n신곡1동,39747,19449,20298,8.64,4.23,4.41,95.82,17136,2.32,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n신곡2동,44657,21531,23126,9.70,4.68,5.02,93.10,18826,2.37,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n송산1동,30633,14946,15687,6.66,3.25,3.41,95.28,13469,2.27,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n송산2동,31499,15582,15917,6.84,3.39,3.46,97.90,13099,2.40,경기도 의정부시청,민원여권과,031-828-2466,2024-09-30\r\n송산3동,44981,21785,23196,9.77

In [86]:
popList = _my.map(lambda x: x.split()).take(3)
print("---00: ", popList[0][0])
print("---01: ", popList[0][1])

---00:  행정기관,인구수(계),인구수(남),인구수(여),구성비(계),구성비(남),구성비(여),성비,세대수,세대당인구,관리기관명,관리부서명,부서전화번호,데이터기준일자
---01:  의정부1동,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,경기도


In [31]:
popList = _my.map(lambda x: x.split()).take(3)
print("---00: ", popList[0][0])
print("---01: ", popList[0])

---00:  행정기관,인구수(계),인구수(남),인구수(여),구성비(계),구성비(남),구성비(여),성비,세대수,세대당인구,관리기관명,관리부서명,부서전화번호,데이터기준일자
---01:  ['행정기관,인구수(계),인구수(남),인구수(여),구성비(계),구성비(남),구성비(여),성비,세대수,세대당인구,관리기관명,관리부서명,부서전화번호,데이터기준일자', '의정부1동,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '의정부2동,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '호원1동,34059,16442,17617,7.40,3.57,3.83,93.33,15178,2.24,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '호원2동,32529,15643,16886,7.07,3.40,3.67,92.64,13272,2.45,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '장암동,18741,8923,9818,4.07,1.94,2.13,90.88,8235,2.28,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '신곡1동,39747,19449,20298,8.64,4.23,4.41,95.82,17136,2.32,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '신곡2동,44657,21531,23126,9.70,4.68,5.02,93.10,18826,2.37,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '송산1동,30633,14946,15687,6.66,3.25,3.41,95.28,13469,2.27,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30',

In [87]:
popList = _my.map(lambda x: x.split()).take(3)
print("---00: ", popList[0][2])
print("---01: ", popList[0][3])

---00:  의정부시청,민원여권과,031-828-2466,2024-09-30
---01:  의정부2동,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,경기도


In [88]:
popList = _my.map(lambda x: x.split()).take(3)

In [89]:
print(popList)

[['행정기관,인구수(계),인구수(남),인구수(여),구성비(계),구성비(남),구성비(여),성비,세대수,세대당인구,관리기관명,관리부서명,부서전화번호,데이터기준일자', '의정부1동,39567,20025,19542,8.60,4.35,4.25,102.47,23371,1.69,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '의정부2동,29644,14758,14886,6.44,3.21,3.23,99.14,16051,1.85,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '호원1동,34059,16442,17617,7.40,3.57,3.83,93.33,15178,2.24,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '호원2동,32529,15643,16886,7.07,3.40,3.67,92.64,13272,2.45,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '장암동,18741,8923,9818,4.07,1.94,2.13,90.88,8235,2.28,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '신곡1동,39747,19449,20298,8.64,4.23,4.41,95.82,17136,2.32,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '신곡2동,44657,21531,23126,9.70,4.68,5.02,93.10,18826,2.37,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '송산1동,30633,14946,15687,6.66,3.25,3.41,95.28,13469,2.27,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', '송산2동,31499,15582,15917,6.84,3.39,3.46,97.90,13099,2.40,경기도', '의정부시청,민원여권과,031-828-2466,2024-09-30', 

In [96]:
#RDD말고 Dataframe으로 읽어오면 더 간단하고 깔끔하다

In [38]:
popDf = spark\
            .read.option("charset", "euc-kr")\
            .option("header", "true")\
            .csv(os.path.join("data","경기도 의정부시_인구현황_20240930.csv"))

In [39]:
popDf.show(5)

+---------+----------+----------+----------+----------+----------+----------+------+------+----------+-----------------+----------+------------+--------------+
| 행정기관|인구수(계)|인구수(남)|인구수(여)|구성비(계)|구성비(남)|구성비(여)|  성비|세대수|세대당인구|       관리기관명|관리부서명|부서전화번호|데이터기준일자|
+---------+----------+----------+----------+----------+----------+----------+------+------+----------+-----------------+----------+------------+--------------+
|의정부1동|     39567|     20025|     19542|      8.60|      4.35|      4.25|102.47| 23371|      1.69|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|의정부2동|     29644|     14758|     14886|      6.44|      3.21|      3.23| 99.14| 16051|      1.85|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|  호원1동|     34059|     16442|     17617|      7.40|      3.57|      3.83| 93.33| 15178|      2.24|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|  호원2동|     32529|     15643|     16886|      7.07|      3.40|      3.67| 92.64| 13272|      2.45|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|   장암동

In [99]:
agedDf = spark\
            .read.option("charset", "euc-kr")\
            .option("header", "true")\
            .csv(os.path.join("data","제주특별자치도 서귀포시_고령화비율및노령화지수현황_20240419.csv"))

In [100]:
agedDf.show(5)

+----+---+---------------+----------------+---------------+----------+----------+--------------+
|년도| 월|서귀포시 인구수|65세이상 인구수 |14세이하 인구수|고령화비율|노령화지수|데이터기준일자|
+----+---+---------------+----------------+---------------+----------+----------+--------------+
|2008| 12|         153120|           22241|          26792|     14.53|     83.01|    2024-04-19|
|2009| 12|         152285|           23031|          25504|     15.12|     90.30|    2024-04-19|
|2010| 12|         153716|           23990|          24633|     15.61|     97.39|    2024-04-19|
|2011| 12|         153366|           24839|          23686|     16.20|    104.87|    2024-04-19|
|2012| 12|         154057|           25826|          22861|     16.76|    112.97|    2024-04-19|
+----+---+---------------+----------------+---------------+----------+----------+--------------+
only showing top 5 rows



In [101]:
%%writefile src/ds3_popCsvRead.py
#!/usr/bin/env python3
# -*- coding: UTF-8 -*-
import os
import pyspark

def doIt():
    print ("---------RESULT-----------")
    popDf = spark\
                .read.option("charset", "euc-kr")\
                .option("header", "true")\
                .csv(os.path.join("data","경기도 의정부시_인구현황_20240930.csv"))
    popDf.show(5)
    agedDf = spark\
                .read.option("charset", "euc-kr")\
                .option("header", "true")\
                .csv(os.path.join("data","제주특별자치도 서귀포시_고령화비율및노령화지수현황_20240419.csv"))
    agedDf.show(5)

if __name__ == "__main__":
    #os.environ["PYSPARK_PYTHON"]="/usr/bin/python3"
    #os.environ["PYSPARK_DRIVER_PYTHON"]="/usr/bin/python3"
    myConf=pyspark.SparkConf()
    spark = pyspark.sql.SparkSession.builder\
        .master("local")\
        .appName("myApp")\
        .config(conf=myConf)\
        .getOrCreate()
    doIt()
    spark.stop()

Overwriting src/ds3_popCsvRead.py


In [188]:
#참고로 위처럼 한 번에 일괄실행하는 방식을 batch ( 빵 구울때 한 판을 지칭 ) 라고 부른다
#그 전에 한줄씩 대화형식으로 코딩하는 방식은 interance

In [189]:
#위의 *-를 쓰는 걸 기호 각각 쉬 뱅이라고 한다 

In [192]:
#main 함수를 지정하면 main함수부터 진행된다는 뜻이다 -> 순차적이 아니라 main이 실행되는 것 그래서 위에서 미리 선언해줘야해

In [193]:
#spark.stop()으로 다 쓰면 메모리에서 제거한다

In [197]:
#참고로 Loc lines of code라고 짧게 할수록 잘하는 것으로 볼 수 있음
#pythonic하다는게 python의 특징이라는 말인데 한줄로 많은 것을 나타낼 수 있기 때문이다 (lamvda같은 걸 많이 쓰기 때문)

In [102]:
!spark-submit src/ds3_popCsvRead.py

---------RESULT-----------
+---------+----------+----------+----------+----------+----------+----------+------+------+----------+-----------------+----------+------------+--------------+
| 행정기관|인구수(계)|인구수(남)|인구수(여)|구성비(계)|구성비(남)|구성비(여)|  성비|세대수|세대당인구|       관리기관명|관리부서명|부서전화번호|데이터기준일자|
+---------+----------+----------+----------+----------+----------+----------+------+------+----------+-----------------+----------+------------+--------------+
|의정부1동|     39567|     20025|     19542|      8.60|      4.35|      4.25|102.47| 23371|      1.69|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|의정부2동|     29644|     14758|     14886|      6.44|      3.21|      3.23| 99.14| 16051|      1.85|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|  호원1동|     34059|     16442|     17617|      7.40|      3.57|      3.83| 93.33| 15178|      2.24|경기도 의정부시청|민원여권과|031-828-2466|    2024-09-30|
|  호원2동|     32529|     15643|     16886|      7.07|      3.40|      3.67| 92.64| 13272|      2.45|경기도 의정부시청|민원여권과|031-828-2

24/11/10 20:47:50 INFO SparkContext: Running Spark version 3.5.3
24/11/10 20:47:50 INFO SparkContext: OS info Windows 11, 10.0, amd64
24/11/10 20:47:50 INFO SparkContext: Java version 21.0.5
24/11/10 20:47:50 INFO ResourceUtils: ==============================================================
24/11/10 20:47:50 INFO ResourceUtils: No custom resources configured for spark.driver.
24/11/10 20:47:50 INFO ResourceUtils: ==============================================================
24/11/10 20:47:50 INFO SparkContext: Submitted application: myApp
24/11/10 20:47:50 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
24/11/10 20:47:50 INFO ResourceProfile: Limiting resource is cpu
24/11/10 20:47:50 INFO ResourceProfileManager: Added ResourceProfile i

In [105]:
#1. map

In [106]:
nRdd = spark.sparkContext.parallelize([1, 2, 3, 4])
squared = nRdd.map(lambda x: x * x)

print (squared)

PythonRDD[133] at RDD at PythonRDD.scala:53


In [107]:
print (squared.collect())

[1, 4, 9, 16]


In [112]:
print (squared.glom().collect())

[[1, 4, 9, 16]]


In [113]:
myRdd4.take(5)

['35, 2', '40, 27', '12, 38', '15, 31', '21, 1']

In [114]:
#''로 표기된 걸 보면 문자형으로 저장되어 있다는 걸 알 수 있다

In [43]:
myRdd5 = myRdd4.map(lambda line: line.split(','))
myRdd5.take(5)

NameError: name 'myRdd4' is not defined

In [44]:
#문자 -> int 정수형으로 형변환

x=['35', ' 2']
y=list()
for i in x:
    y.append(int(i))
print(y)

[35, 2]


In [45]:
int_Rdd = list()


for i in myRdd5:
    int_Rdd.append(int(i))

print(int_Rdd)

NameError: name 'myRdd5' is not defined

In [46]:
myRdd6 = myRdd5.map(lambda x: [int(i) for i in x])
myRdd6.take(5)

#lambda 옆에 있는 x는 .map 앞에 있는 myRdd5의 값을 받은 거!

NameError: name 'myRdd5' is not defined

In [42]:
myRdd2=spark.sparkContext\
    .textFile(os.path.join("data","ds_spark_wiki.txt"))

In [123]:
sentences=myRdd2.map(lambda x:x.split())

In [124]:
sentences.count()

10

In [125]:
def mySplit(x):
    return x.split()

sentences2=myRdd2.map(mySplit)
sentences2.count()

10

In [126]:
sentences.take(3)

[['Wikipedia'],
 ['Apache',
  'Spark',
  'is',
  'an',
  'open',
  'source',
  'cluster',
  'computing',
  'framework.'],
 ['아파치', '스파크는', '오픈', '소스', '클러스터', '컴퓨팅', '프레임워크이다.']]

In [127]:
# .map는 한 줄씩 읽는 거고
# .count는 단어의 갯수를 읽는다

In [135]:
sentences2.take(20)
#10개까지 밖에 없어서 10부터는 출력되는 값이 동일함

[['Wikipedia'],
 ['Apache',
  'Spark',
  'is',
  'an',
  'open',
  'source',
  'cluster',
  'computing',
  'framework.'],
 ['아파치', '스파크는', '오픈', '소스', '클러스터', '컴퓨팅', '프레임워크이다.'],
 ['Apache', 'Spark', 'Apache', 'Spark', 'Apache', 'Spark', 'Apache', 'Spark'],
 ['아파치', '스파크', '아파치', '스파크', '아파치', '스파크', '아파치', '스파크'],
 ['Originally',
  'developed',
  'at',
  'the',
  'University',
  'of',
  'California,',
  "Berkeley's",
  'AMPLab,'],
 ['the',
  'Spark',
  'codebase',
  'was',
  'later',
  'donated',
  'to',
  'the',
  'Apache',
  'Software',
  'Foundation,'],
 ['which', 'has', 'maintained', 'it', 'since.'],
 ['Spark',
  'provides',
  'an',
  'interface',
  'for',
  'programming',
  'entire',
  'clusters',
  'with'],
 ['implicit', 'data', 'parallelism', 'and', 'fault-tolerance.']]

In [128]:
for line in sentences.collect():
    for word in line:
        print (word, end=" ")
    print ("\n-----")

Wikipedia 
-----
Apache Spark is an open source cluster computing framework. 
-----
아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다. 
-----
Apache Spark Apache Spark Apache Spark Apache Spark 
-----
아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크 
-----
Originally developed at the University of California, Berkeley's AMPLab, 
-----
the Spark codebase was later donated to the Apache Software Foundation, 
-----
which has maintained it since. 
-----
Spark provides an interface for programming entire clusters with 
-----
implicit data parallelism and fault-tolerance. 
-----


In [136]:
myRdd2.map(lambda s:len(s)).collect()

[9, 59, 32, 51, 31, 72, 71, 30, 64, 46]

In [137]:
myList=["this is","a line"]
_rdd=spark.sparkContext.parallelize(myList)

In [138]:
repRdd=_rdd.map(lambda x:x.replace("this","This"))
repRdd.take(10)

['This is', 'a line']

In [139]:
's'.upper()

'S'

In [140]:
wordsRdd=_rdd.map(lambda x:x.split())
print (wordsRdd.collect())

[['this', 'is'], ['a', 'line']]


In [141]:
upperRDD =wordsRdd.map(lambda x: x[0].upper())
print (upperRDD.collect())

['THIS', 'A']


In [142]:
upperRDD =wordsRdd.map(lambda x: x[1].upper())
print (upperRDD.collect())

['IS', 'LINE']


In [143]:
upper2RDD =wordsRdd.map(lambda x: [i.upper() for i in x])
print (upper2RDD.collect())

[['THIS', 'IS'], ['A', 'LINE']]


In [146]:
myRdd100 = spark.sparkContext.parallelize(range(1,101))
myRdd100.reduce(lambda subtotal, x: subtotal + x)

5050

In [147]:
spark.sparkContext.parallelize(range(1,11),2).glom().collect()

[[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]]

In [148]:
spark.sparkContext.parallelize(range(1,11)).fold(0, lambda subtotal, x: subtotal + x)

55

In [149]:
spark.sparkContext.parallelize(range(1,11),1).fold(10, lambda subtotal, x: subtotal + x)

75

In [150]:
spark.sparkContext.parallelize(range(1,11),2).fold(10, lambda subtotal, x: subtotal + x)

85

In [151]:
spark.sparkContext.parallelize(range(1,11),5).fold(0, lambda subtotal, x: subtotal + x)

55

In [154]:
print ("sum: ", myRdd100.sum())
print ("min: ", myRdd100.min())
print ("max: ", myRdd100.max())
print ("count: ", myRdd100.count())
print ("standard deviation:", myRdd100.stdev())
print ("variance: ", myRdd100.variance())

sum:  5050
min:  1
max:  100
count:  100
standard deviation: 28.86607004772212
variance:  833.25


In [153]:
#fold는 리듀스랑 거의 비슷한데 초기값을 지정한 값으로 계속 더해준다는 것이 특징

In [155]:
#넘파이는 함수를 이용하여 바로 쓸 수 있다 (배열프로그래밍 이기 때문)

In [161]:
#filter() 함수를 써보자

# 아래는 "Spark" 단어가 포함된 문장이 조건으로 spark가 들어있는 문장이 몇개가 있는지를 알려준다
# count를 이용하여 갯수를 확인할 수 있다

In [162]:
myRdd_spark=myRdd2.filter(lambda line: "Spark" in line)
print ("How many lines having 'Spark': ", myRdd_spark.count())
#filter은 분리!

How many lines having 'Spark':  4


In [163]:
myRdd2

data\ds_spark_wiki.txt MapPartitionsRDD[143] at textFile at NativeMethodAccessorImpl.java:0

In [165]:
myRdd2.take(20)

['Wikipedia',
 'Apache Spark is an open source cluster computing framework.',
 '아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.',
 'Apache Spark Apache Spark Apache Spark Apache Spark',
 '아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크',
 "Originally developed at the University of California, Berkeley's AMPLab,",
 'the Spark codebase was later donated to the Apache Software Foundation,',
 'which has maintained it since.',
 'Spark provides an interface for programming entire clusters with',
 'implicit data parallelism and fault-tolerance.']

In [172]:
#한글도 지원 된다
#다만 유니코드형식으로 구성되어 있지 않다면 u를 붙여줘야함 

myRdd_unicode = myRdd2.filter(lambda line: u"스파크" in line)
print (myRdd_unicode.first())
print (" === ")
print (myRdd_unicode.take(4))

아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.
 === 
['아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.', '아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크']


In [167]:
myRdd_spark=myRdd2.filter(lambda line: "Spark" in line)

In [198]:
#flatMap() 함수의 기능
#2차원 배열 -> 1차원 배열

In [169]:
stopwords = ['is','am','are','the','for','a', 'an', 'at']
myRdd_stop = myRdd2.flatMap(lambda x:x.split())\
                    .filter(lambda x: x not in stopwords)

In [170]:
for words in myRdd_stop.collect():
    print (words, end=' ')

Wikipedia Apache Spark open source cluster computing framework. 아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다. Apache Spark Apache Spark Apache Spark Apache Spark 아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크 Originally developed University of California, Berkeley's AMPLab, Spark codebase was later donated to Apache Software Foundation, which has maintained it since. Spark provides interface programming entire clusters with implicit data parallelism and fault-tolerance. 

In [178]:
#filter를 이용하여 특정 문자를 포함한 경우는 제외할 수 있다!

stopwords = ['is','am','are','the','for','a', 'an', 'at']

myRdd_stop = myRdd2.flatMap(lambda x:x.split())\
                    .filter(lambda x: x not in stopwords)

for words in myRdd_stop.collect():
    print (words, end=' ')

Wikipedia Apache Spark open source cluster computing framework. 아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다. Apache Spark Apache Spark Apache Spark Apache Spark 아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크 Originally developed University of California, Berkeley's AMPLab, Spark codebase was later donated to Apache Software Foundation, which has maintained it since. Spark provides interface programming entire clusters with implicit data parallelism and fault-tolerance. 

In [200]:
#반환값을 주지 않는 action , 로그에 기록하는 함수
# <foreach>

In [182]:
spark.sparkContext.parallelize([1, 2, 3, 4, 5]).foreach(lambda x: x + 1)

In [183]:
spark.sparkContext.parallelize([1, 2, 3, 4, 5]).map(lambda x: x + 1).collect()

[2, 3, 4, 5, 6]

In [184]:
def f(x): print(x)
spark.sparkContext.parallelize([1, 2, 3, 4, 5]).foreach(f)

In [202]:
#파이프라인 pipeline => 계속 붙여서 연이어 적용하는 방식
#trans - action을 한꺼번에 하는, 명령어가 연결되는 방식! 사실 지금까지 써왔음 

In [186]:
upper2list=wordsRdd.map(lambda x: [i.upper() for i in x]).collect()
print (upper2list)

[['THIS', 'IS'], ['A', 'LINE']]


In [187]:
wordsLength = wordsRdd\
    .map(len)\
    .collect()
print (wordsLength)

[2, 2]


In [203]:
###############파일에서 쓰기는 생략합니다#################

In [204]:
#myRdd_group=myRdd2.groupBy(lambda x:x[0:2])
myRdd_group=myRdd2.groupBy(lambda x:"아파치" in x)

for (k,v) in myRdd_group.collect():
    print ("{}: {}".format(k, v))

False: <pyspark.resultiterable.ResultIterable object at 0x000001FA69202310>
True: <pyspark.resultiterable.ResultIterable object at 0x000001FA692BB1F0>


In [205]:
#myRdd_group=myRdd2.flatMap(lambda x:x.split()).groupBy(lambda x:w[0:2])
#myRdd_group=myRdd2.groupBy(lambda x:x[0:2])

for (k,v) in myRdd_group.collect():
    for eachValue in v:
        print ("{}: {}".format(k, eachValue))
    print ("-----")

False: Wikipedia
False: Apache Spark is an open source cluster computing framework.
False: Apache Spark Apache Spark Apache Spark Apache Spark
False: Originally developed at the University of California, Berkeley's AMPLab,
False: the Spark codebase was later donated to the Apache Software Foundation,
False: which has maintained it since.
False: Spark provides an interface for programming entire clusters with
False: implicit data parallelism and fault-tolerance.
-----
True: 아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.
True: 아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크
-----


In [206]:
#groupBy 매우 중요함
#특히 통계에서 집단화를 하는 것이 중요
#통찰력을 높이기 위함
#ex) 남학생별, 학년별 등등..

In [207]:
#transformation함수인 것

In [208]:
#paird unpaird가 쌍으로 되어있거나 (딕셔너리) 그렇지 않거나를 말한다

In [210]:
myRdd2.take(10)

['Wikipedia',
 'Apache Spark is an open source cluster computing framework.',
 '아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.',
 'Apache Spark Apache Spark Apache Spark Apache Spark',
 '아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크',
 "Originally developed at the University of California, Berkeley's AMPLab,",
 'the Spark codebase was later donated to the Apache Software Foundation,',
 'which has maintained it since.',
 'Spark provides an interface for programming entire clusters with',
 'implicit data parallelism and fault-tolerance.']

In [211]:
#myRdd_group=myRdd2.groupBy(lambda x:x[0:2])
myRdd_group=myRdd2.groupBy(lambda x:"아파치" in x)

for (k,v) in myRdd_group.collect():
    print ("{}: {}".format(k, v))


#한줄씩이 x고, 아파치가 있는 그룹과 없는 그룹으로 나누겠다는 뜻

False: <pyspark.resultiterable.ResultIterable object at 0x000001FA692BB220>
True: <pyspark.resultiterable.ResultIterable object at 0x000001FA692AC6D0>


In [214]:
#data를 어떤 식으로든 가공할 수 있어야 한다
#위처럼 하면 T/F두 그룹으로 나눠질 수 있겠다고 예측 가능

#myRdd_group=myRdd2.flatMap(lambda x:x.split()).groupBy(lambda x:w[0:2])
#myRdd_group=myRdd2.groupBy(lambda x:x[0:2])

#있는지 없는지가 집단화명(True/False)

for (k,v) in myRdd_group.collect():
    for eachValue in v:
        print ("{}: {}".format(k, eachValue))
    print ("-----")

False: Wikipedia
False: Apache Spark is an open source cluster computing framework.
False: Apache Spark Apache Spark Apache Spark Apache Spark
False: Originally developed at the University of California, Berkeley's AMPLab,
False: the Spark codebase was later donated to the Apache Software Foundation,
False: which has maintained it since.
False: Spark provides an interface for programming entire clusters with
False: implicit data parallelism and fault-tolerance.
-----
True: 아파치 스파크는 오픈 소스 클러스터 컴퓨팅 프레임워크이다.
True: 아파치 스파크 아파치 스파크 아파치 스파크 아파치 스파크
-----
